# Lab 1｜从零写 Two-Tower + in-batch loss

原理部分在 `docs/01-two-tower/` 的五篇正文里。这里是动手部分：写两座塔和 batch 内交叉熵，然后在真实数据上跑一次参数更新。

格子开头的符号只是提示：📖 概念，🧩 要你写的代码，✅ 断言，🚀 可以改着玩的地方，💡 折叠的参考答案。

两个提醒：遇到末尾带问题的格子，先写一句再往下跑；`TwoTower` 这个类后面还会改，改完记得重跑「类定义 → 建模型」这两格。

In [ ]:
# 不用改，直接运行
import math
import sys
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F


def find_labs_dir() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / 'labs'):
            if (candidate / 'labkit.py').exists():
                return candidate
    raise FileNotFoundError('找不到 labs 目录，请从仓库根目录或 labs 目录启动')


LABS_DIR = find_labs_dir()
if str(LABS_DIR) not in sys.path:
    sys.path.insert(0, str(LABS_DIR))

import labkit  # noqa: E402

PASSED = []


def check(name: str, actual, expected) -> None:
    '''严格相等：列表、元组、整数、布尔。'''
    ok = actual == expected
    print(('PASS' if ok else 'FAIL'), name, '| 期望:', expected, '| 实际:', actual)
    if not ok:
        raise AssertionError('还没对，改那一格，或看下面的参考答案')
    PASSED.append(name)


def check_close(name: str, actual: float, expected: float, tolerance: float = 1e-5) -> None:
    '''浮点比较，允许误差。'''
    ok = abs(actual - expected) <= tolerance
    print(('PASS' if ok else 'FAIL'), name, '| 期望约:', expected, '| 实际:', actual)
    if not ok:
        raise AssertionError('还没对，改那一格，或看下面的参考答案')
    PASSED.append(name)


print('labs dir:', LABS_DIR)
print('torch', torch.__version__)

## 📖 理解一下：两座塔，两张表

要回答的问题就一个：用户 u 有多喜欢物品 i。

最省事的做法是给每个用户、每个物品各学一个向量，让两者落在同一个空间里，可以直接比较：

```text
user u  ->  vector_u  [D]
item i  ->  vector_i  [D]
score   =  vector_u · vector_i        # 内积，越大越喜欢
```

用户和物品是两套互不相干的编号（用户 0 和电影 0 没有关系），所以要两张独立的 Embedding 表。

至于「塔」这个名字：实际系统里两侧通常不是裸查表，而是各自接一段网络（MLP、序列模型等等）。现在先只做查表。

两个问题：

1. 如果只用一张表，让用户和物品共用同一批行，会出什么问题？
2. 两张表的宽度（`embedding_dim`）必须一样吗？为什么？

> 完整推导与文献：[01｜数据与共享空间](../docs/01-two-tower/01-data-and-shared-space.md)

### 🧩 你来写：`TwoTower` 的四处

- `TODO 1-0`：两张独立的表
- `TODO 1-1` / `TODO 1-2`：`encode_users()` / `encode_items()`，查表后做 L2 normalize
- `TODO 1-3`：`forward()` 里让 B 个用户和 B 个物品两两打分（temperature 先不管）

改完运行这一格和下面两格。

In [ ]:
class TwoTower(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim, temperature=1.0):
        super().__init__()
        if temperature <= 0:
            raise ValueError('temperature 必须为正')
        # TODO 1-0：两张独立的表，再把 temperature 存起来
        #   提示：nn.Embedding(实体数量, 向量维度)
        #   num_users 行、num_items 行，宽度都是 embedding_dim
        raise NotImplementedError('TODO 1-0')

    def encode_users(self, user_ids):
        # TODO 1-1：查表得到 [B, D]，再做 L2 normalize（dim=1）
        #   提示：F.normalize(vectors, p=2, dim=1)
        raise NotImplementedError('TODO 1-1')

    def encode_items(self, item_ids):
        # TODO 1-2：和 1-1 一样，只是换成物品表
        raise NotImplementedError('TODO 1-2')

    def forward(self, user_ids, item_ids):
        user_vectors = self.encode_users(user_ids)
        item_vectors = self.encode_items(item_ids)
        # TODO 1-3：B 个用户和 B 个物品两两打分，返回 [B, B]（temperature 先不管）
        raise NotImplementedError('TODO 1-3')

In [ ]:
# 不用改：造一个玩具模型，先用 4 对交互把逻辑跑通
torch.manual_seed(42)

TOY_USERS, TOY_ITEMS, DIM = 10, 20, 64
TOY_B = 4

toy_model = TwoTower(TOY_USERS, TOY_ITEMS, embedding_dim=DIM, temperature=1.0)
toy_user_ids = torch.tensor([0, 1, 2, 3])
toy_item_ids = torch.tensor([4, 5, 6, 7])

print('user_ids:', toy_user_ids.tolist())
print('item_ids:', toy_item_ids.tolist())
print('每个用户配一个他喜欢的物品，正好 4 对。')

In [ ]:
# ✅ 跑一下：形状、向量长度、参数量
toy_user_vectors = toy_model.encode_users(toy_user_ids)
toy_item_vectors = toy_model.encode_items(toy_item_ids)

check('1-1 用户向量形状', list(toy_user_vectors.shape), [TOY_B, DIM])
check('1-1 物品向量形状', list(toy_item_vectors.shape), [TOY_B, DIM])

user_norms = toy_user_vectors.norm(dim=1)
item_norms = toy_item_vectors.norm(dim=1)
check_close('1-1 第一个用户向量长度接近 1', user_norms[0].item(), 1.0)
check(
    '1-1 四个用户向量长度都接近 1',
    bool(torch.allclose(user_norms, torch.ones_like(user_norms), atol=1e-5)),
    True,
)
check(
    '1-1 四个物品向量长度都接近 1',
    bool(torch.allclose(item_norms, torch.ones_like(item_norms), atol=1e-5)),
    True,
)

check(
    '1-1 参数量 = (num_users + num_items) * D',
    sum(parameter.numel() for parameter in toy_model.parameters()),
    (TOY_USERS + TOY_ITEMS) * DIM,
)

<details>
<summary>💡 参考答案（<code>TwoTower</code>，先不带 temperature）</summary>

```python
class TwoTower(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim, temperature=1.0):
        super().__init__()
        if temperature <= 0:
            raise ValueError('temperature 必须为正')
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.temperature = temperature

    def encode_users(self, user_ids):
        user_vectors = self.user_embedding(user_ids)          # [B, D]
        return F.normalize(user_vectors, p=2, dim=1)          # 长度接近 1

    def encode_items(self, item_ids):
        item_vectors = self.item_embedding(item_ids)          # [B, D]
        return F.normalize(item_vectors, p=2, dim=1)

    def forward(self, user_ids, item_ids):
        user_vectors = self.encode_users(user_ids)            # [B, D]
        item_vectors = self.encode_items(item_ids)            # [B, D]
        return user_vectors @ item_vectors.T                  # [B, B]
```

</details>

容易写错的两处：`nn.Embedding` 的参数顺序（先实体数量，再向量维度），以及 normalize 用 `dim=1` 而不是 `dim=0`——`[B, D]` 里每一行是一个向量。

## 📖 理解一下：一个 batch 变成一张 `B×B` 的表

`DataLoader` 送来的是一个 batch 的正样本对：

```text
user_ids = [u0, u1, u2, u3]      # [B]
item_ids = [i0, i1, i2, i3]      # [B]，第 k 项是 u_k 喜欢过的物品
```

两个向量矩阵相乘：

```text
[B, D] @ [D, B]  ->  [B, B]
```

第 `i` 行第 `j` 列就是用户 i 对物品 j 的分数。只编码 B 个用户和 B 个物品，就换到 B×B 个训练信号——这是 Two-Tower 省的地方；矩阵本身还是 O(B²) 的计算和显存，所以 B 不能无限大。

先猜一下：第 0 行那 4 个分数里哪个最大？是第 0 列（用户 0 自己的正样本）吗？

换个说法：模型刚初始化，参数还是随机的，凭什么让用户 0 的正样本排第一？

In [ ]:
# ✅ 跑一下：形状是不是 [B, B]，每行最大值落在哪
torch.set_printoptions(precision=3, sci_mode=False)

toy_logits = toy_model(toy_user_ids, toy_item_ids)
check('1-2 logits 形状是 [B, B]', list(toy_logits.shape), [TOY_B, TOY_B])

print('相似度矩阵:')
print(toy_logits.detach())
print('每一行最大的列:', toy_logits.detach().argmax(dim=1).tolist())
print()
print('现在 temperature=1.0，所以 logits 就是 cosine similarity，落在 [-1, 1]。')
print('随机初始化下，每一行的最大值基本不会正好落在对角线上。')

## 📖 理解一下：对角线才是「正确答案」

输入是按正样本对对齐的，所以正确答案的位置是固定的：

```text
row 0: 用户 0 的正样本是 item_ids[0]，它排在第 0 列   ->  正确答案 = 第 0 列
row 1: 用户 1 的正样本是 item_ids[1]，它排在第 1 列   ->  正确答案 = 第 1 列
row 2: ...                                        ->  正确答案 = 第 2 列
row 3: ...                                        ->  正确答案 = 第 3 列
```

对角线当正样本、其他列当负样本，这个做法叫 **in-batch negatives**。省掉了单独采样负样本的工程，代价是一个还没验证的假设：

> 同一批里其他用户的物品，对这个用户来说都是负样本吗？

这里先留个问题，Lab 3 会用真实数据量一下。

另外注意第 0 行的 label 不是电影 ID，是列号 0。

`B = 4` 时，`F.cross_entropy` 要的 `targets` 是什么？
（提示：长度 4 的向量，`targets[i]` 是第 `i` 行的列号）

> 完整推导与文献：[02｜一个 batch 怎么变成可训练的目标](../docs/01-two-tower/02-in-batch-objective.md)

### 🧩 你来写：`in_batch_targets()`

返回长度为 `batch_size` 的 target，第 `i` 个元素是 `i`。

In [ ]:
def in_batch_targets(batch_size):
    '''返回长度为 batch_size 的 target：第 i 个元素是 i。'''
    # TODO 1-4：提示 torch.arange
    raise NotImplementedError('TODO 1-4')

In [ ]:
# ✅ 跑一下：B=4/1/8 时 targets 是不是 0,1,2,…
check('1-3 targets(4)', in_batch_targets(4).tolist(), [0, 1, 2, 3])
check('1-3 targets(1)', in_batch_targets(1).tolist(), [0])
check('1-3 targets(8)', in_batch_targets(8).tolist(), [0, 1, 2, 3, 4, 5, 6, 7])

<details>
<summary>💡 参考答案（<code>in_batch_targets</code>）</summary>

```python
def in_batch_targets(batch_size):
    '''返回长度为 batch_size 的 target：第 i 个元素是 i。'''
    return torch.arange(batch_size)
```

</details>

为什么不是 `item_ids`：`logits` 是 `[B, B]`，列只有 `0 ~ B-1` 这 B 个位置。`cross_entropy` 要的是「第几列」，不是「哪个物品」。塞电影 ID 进去会越界，下一格能看到。

## 📖 理解一下：`F.cross_entropy` 在这里做了什么

它对每一行做三件事：

1. 把这一行的 B 个分数做 softmax，变成概率；
2. 取出 `target[i]` 那一列的概率；
3. 取负对数，再对所有行求平均。

也就是 `loss = -mean( log_softmax(logits)[i, target[i]] )`。输入里没有「电影 ID」这个概念，只有列号。

猜一个数：如果模型什么都没学到，每一行的 4 个分数完全一样（比如全是 0），softmax 之后每列都是 `1/4`，这时候 loss 是多少？（提示：`-log(1/4) = log(4)`）

### 🧩 你来写：`in_batch_cross_entropy_loss()`

两行：造出 target 列号，交给 `F.cross_entropy`。

In [ ]:
def in_batch_cross_entropy_loss(logits):
    '''logits: [B, B]，第 i 行的正类在第 i 列。返回标量 loss。'''
    batch_size = logits.shape[0]
    # TODO 1-5：造出 target 列号交给 F.cross_entropy，device 与 logits 保持一致
    raise NotImplementedError('TODO 1-5')

In [ ]:
# ✅ 跑一下：是不是标量、和手算是否一致、均匀分数时是不是 ln(B)
toy_loss = in_batch_cross_entropy_loss(toy_logits)
check('1-4 loss 是标量', tuple(toy_loss.shape), ())

# 手算一遍：取每行对角线，做 log_softmax 后取负、求平均
manual = -F.log_softmax(toy_logits.detach(), dim=1)[
    torch.arange(TOY_B), torch.arange(TOY_B)
].mean()
check_close('1-4 与手算的 log_softmax 一致', toy_loss.item(), manual.item())

# 分数完全均匀时，loss 必须恰好是 ln(B)
uniform_loss = in_batch_cross_entropy_loss(torch.zeros(TOY_B, TOY_B))
check_close('1-4 分数均匀时 loss == ln(B)', uniform_loss.item(), math.log(TOY_B))
print('参考：ln(4) =', round(math.log(TOY_B), 6), '| 真实训练用 B=256，ln(256) =', round(math.log(256), 4))

<details>
<summary>💡 参考答案（<code>in_batch_cross_entropy_loss</code>）</summary>

```python
def in_batch_cross_entropy_loss(logits):
    '''logits: [B, B]，第 i 行的正类在第 i 列。返回标量 loss。'''
    batch_size = logits.shape[0]
    targets = torch.arange(batch_size, device=logits.device)
    return F.cross_entropy(logits, targets)
```

</details>

容易漏的是 `device=logits.device`。在 CPU 上跑看不出区别，等模型搬到 GPU，target 留在 CPU 就会报 device mismatch。

In [ ]:
# 🚀 试试看：把 target 写成电影 ID
# 这格故意写错，先猜报什么错再运行。
try:
    F.cross_entropy(toy_logits.detach(), toy_item_ids)
except Exception as error:
    print('反例 ->', type(error).__name__, ':', error)
    print()
    print('logits 是 [4,4]，合法的列号只有 0~3；而 item_ids 是 [4,5,6,7]，第一个 4 就越界了。')
    print('所以 target 是「第几列」，不是「哪个 item」。')

## 📖 理解一下：temperature 的作用

L2 normalize 之后 cosine similarity 只能落在 `[-1, 1]`。随机初始化时这些分数挤在一起，softmax 几乎是均匀的，正负拉不开。

除以一个小于 1 的数可以把差异放大：

```text
相似度 [0.42, 0.38, 0.35, 0.40]      softmax 最大概率约 0.26
除以 0.1 后 [4.20, 3.80, 3.50, 4.00]  softmax 明显更尖
```

它主要作用在训练上：logit 的尺度决定了 softmax 有多尖锐，也决定了梯度的尺度——分数先除以 T，反向传播时这个 `1/T` 会乘回梯度上。

检索那边不受影响：只看排序时，所有分数乘同一个正数不改变名次。所以缩放放在 `forward` 还是 loss 里都行，工程实现里统一放在 `forward`，取 `temperature=0.1`。（这是就排序名次而言；训练时用不同 T 会学出不同参数，最终指标仍可能不同。）

temperature 从 1.0 降到 0.1，softmax 会变平还是变尖？

> 完整推导与文献：[02｜一个 batch 怎么变成可训练的目标](../docs/01-two-tower/02-in-batch-objective.md)

### 🧩 你来写：给 `forward` 加上 temperature

回到上面的 `TwoTower`，把 `forward` 的返回改成 `相似度矩阵 / self.temperature`。

改完按顺序重跑：类定义格 → 建模型格 → 下面这格。

In [ ]:
# ✅ 跑一下：logits 是不是恰好等于 相似度 / 0.1，softmax 有没有变尖
temperature_model = TwoTower(TOY_USERS, TOY_ITEMS, embedding_dim=DIM, temperature=0.1)

raw_similarity = temperature_model.encode_users(toy_user_ids) @ temperature_model.encode_items(
    toy_item_ids
).T
temperature_logits = temperature_model(toy_user_ids, toy_item_ids)

check_close(
    '1-5 temperature=0.1 时 logits == 相似度 / 0.1',
    temperature_logits[0, 0].item(),
    raw_similarity[0, 0].item() / 0.1,
)

flat = F.softmax(raw_similarity[0], dim=0).max().item()
peaked = F.softmax(temperature_logits[0], dim=0).max().item()
print('第 0 行 softmax 最大概率：相似度直接进 =', round(flat, 4), '| 除以 0.1 后 =', round(peaked, 4))
check('1-5 温度变小，softmax 更尖', peaked > flat, True)

<details>
<summary>💡 参考答案（<code>TwoTower</code> 第二版：只有 <code>forward</code> 的返回变了）</summary>

```python
class TwoTower(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim, temperature=1.0):
        super().__init__()
        if temperature <= 0:
            raise ValueError('temperature 必须为正')
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.temperature = temperature

    def encode_users(self, user_ids):
        user_vectors = self.user_embedding(user_ids)          # [B, D]
        return F.normalize(user_vectors, p=2, dim=1)

    def encode_items(self, item_ids):
        item_vectors = self.item_embedding(item_ids)          # [B, D]
        return F.normalize(item_vectors, p=2, dim=1)

    def forward(self, user_ids, item_ids):
        user_vectors = self.encode_users(user_ids)            # [B, D]
        item_vectors = self.encode_items(item_ids)            # [B, D]
        similarity = user_vectors @ item_vectors.T            # [B, B]
        return similarity / self.temperature                  # <- 只多了这里
```

</details>

放在 `forward` 还是 loss 里都可以：训练时它改变 softmax 的尺度，检索时所有分数乘同一个正数，Top-K 名次不变。工程实现里统一放在 `forward`。

## 📖 理解一下：梯度只回到被碰过的行

`loss.backward()` 之后两张表的 `.grad` 都有值，但只有这个 batch 碰过的那几行有值。

因为这个 batch 只出现过 4 个用户、4 个物品，其余的行没进计算图。

Two-Tower 省算力的另一点也在这里：一次 forward 只用到 `2B` 行 embedding，不是全部用户和物品。

In [ ]:
# ✅ 跑一下：两张表的梯度，以及没被碰过的行是不是 0
# 每次都重新 forward，这样这一格可以反复运行
temperature_model.zero_grad(set_to_none=True)
fresh_logits = temperature_model(toy_user_ids, toy_item_ids)
in_batch_cross_entropy_loss(fresh_logits).backward()

check('1-6 用户表有梯度', temperature_model.user_embedding.weight.grad is not None, True)
check('1-6 物品表有梯度', temperature_model.item_embedding.weight.grad is not None, True)
check(
    '1-6 没被碰过的用户梯度为 0',
    temperature_model.user_embedding.weight.grad[TOY_B:].abs().sum().item(),
    0.0,
)
print(
    '收到梯度的用户行:',
    temperature_model.user_embedding.weight.grad.abs().sum(dim=1).nonzero().flatten().tolist(),
)
print('这个 batch 用到的用户:', sorted(set(toy_user_ids.tolist())))

## 📖 理解一下：从玩具张量到真实数据

上面用的是 10 个用户、20 个物品的玩具模型。真实训练没有别的区别，就是把 `num_users/num_items` 换成真实规模，4 对换成 256 对。

MovieLens-1M 原始是 6,040 个用户。按 `rating >= 4` 筛完正反馈，进入模型的是 6,038 个用户、3,533 部电影，训练交互 569,231 条；严格时间边界下 3,025 个用户参与评测，其余用户全部留在 train。

这些 `labkit` 都处理好了：下载、正反馈筛选、ID 编码、按时间切分、`DataLoader`。剩下最后一步，把一次参数更新写出来。

In [ ]:
# 不用改：真实数据 + 一个真实 batch
labkit.set_seed(42)
data = labkit.prepare_movielens()
print(data.describe())

train_loader = labkit.create_train_loader(data.train, batch_size=256, seed=42)
real_batch = next(iter(train_loader))

print('user_idx 形状:', list(real_batch['user_idx'].shape))
print('item_idx 形状:', list(real_batch['item_idx'].shape))
print('第一个 batch 的前 8 个 user_idx:', real_batch['user_idx'][:8].tolist())
print('第一个 batch 的前 8 个 item_idx:', real_batch['item_idx'][:8].tolist())

real_model = TwoTower(data.num_users, data.num_items, embedding_dim=64, temperature=0.1)
real_optimizer = torch.optim.Adam(real_model.parameters(), lr=1e-3)
print('真实模型参数量:', sum(parameter.numel() for parameter in real_model.parameters()))

### 🧩 你来写：`train_one_step()`

五步：forward → 算 loss → `zero_grad` → `backward` → `step`，返回这一步的 loss（float）。

真实训练 `B = 256`，分数完全均匀时 loss 是 `ln(256) = 5.545`。不过 `temperature=0.1` 把分数放大了，这一步实际 loss 会比 5.545 大还是小？先写下来。

In [ ]:
def train_one_step(model, batch, optimizer):
    '''取一个 batch 做一次参数更新，返回这一步的 loss（float）。'''
    user_ids = batch['user_idx']
    item_ids = batch['item_idx']
    # TODO 1-6：五步 —— forward / 算 loss / zero_grad / backward / step
    #   用 .item() 返回 float
    raise NotImplementedError('TODO 1-6')

In [ ]:
# ✅ 跑一下：参数有没有被更新、loss 是不是有限、和 ln(256) 差多少
parameters_before = real_model.user_embedding.weight.detach().clone()
real_loss = train_one_step(real_model, real_batch, real_optimizer)

check(
    '1-7 参数确实被更新了',
    bool(torch.equal(parameters_before, real_model.user_embedding.weight.detach())),
    False,
)
check('1-7 loss 是有限值', bool(torch.isfinite(torch.tensor(real_loss))), True)

print()
print('这一步 loss        =', round(real_loss, 4))
print('完全均匀时应该是   =', round(math.log(256), 4), '（ln(256)）')
print('两个数为什么不相等，就是刚才那个预测的答案。')

<details>
<summary>💡 参考答案（<code>train_one_step</code>）</summary>

```python
def train_one_step(model, batch, optimizer):
    '''取一个 batch 做一次参数更新，返回这一步的 loss（float）。'''
    user_ids = batch['user_idx']
    item_ids = batch['item_idx']

    logits = model(user_ids, item_ids)
    loss = in_batch_cross_entropy_loss(logits)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()
```

</details>

顺序有讲究：`zero_grad()` 要在 `backward()` 之前，否则梯度会累加到上一步；`step()` 要在 `backward()` 之后，否则梯度还没算出来。

In [ ]:
# 💾 把这一章写的代码存成模块，Lab 2 直接 import 它
# 导出的是格子里的原文，不是参考答案
labkit.harvest(
    'myrec/model.py',
    TwoTower,
    in_batch_targets,
    in_batch_cross_entropy_loss,
    train_one_step,
    imports=[
        'import torch',
        'from torch import nn',
        'from torch.nn import functional as F',
    ],
)

# 顺手确认一下导出能用：像 Lab 2 那样 import 一次
import importlib

import myrec.model as my_model

importlib.reload(my_model)

reloaded = my_model.TwoTower(10, 20, embedding_dim=8, temperature=0.1)
check(
    '交接成功：myrec.model 里的 TwoTower 能直接跑',
    list(reloaded(torch.tensor([0]), torch.tensor([1])).shape),
    [1, 1],
)

## 收尾

### 这个 lab 写了多少

| 自己写的 | 行数 | 脚手架给的 |
|---|---|---|
| 两张 Embedding 表 | 3 | `labkit.py` 的数据管道（下载/编码/切分/DataLoader） |
| `encode_users` / `encode_items` | 4 | `check()` / `check_close()` |
| `forward`（含 temperature） | 2 | 建 toy 模型、取真实 batch |
| `in_batch_targets` / `in_batch_cross_entropy_loss` | 3 | 玩具张量和真实数据的规模 |
| `train_one_step` | 6 | |

加起来 18 行左右。Two-Tower 的训练部分就是这些。

### 这几条应该能直接答上来

1. 为什么要两张 Embedding 表？两张表的宽度可以不同吗？
2. `targets` 为什么是 `torch.arange(B)`，不是 `item_ids`？
3. 分数均匀时 loss 为什么正好是 `ln(B)`？
4. temperature 变小，softmax 怎么变？它对 Top-K 检索有影响吗？
5. `backward()` 之后为什么只有几行 embedding 有梯度？

答着不顺就回头看对应的那几格。

### 还没解决的问题

概念 3 那个问题这里不回答：in-batch 的其他列对这个用户来说都是负样本吗？Lab 3 会用真实数据量出比例。

### 下一个 lab

**Lab 2｜接 MovieLens 训练 + full-ranking evaluation**：把 `train_one_step` 扩成一个 epoch 训出第一个模型，再写 Top-K 生成、seen-item 过滤和 Recall/NDCG。从「loss 会下降」走到「推荐准不准」。